> **MIT License** - Copyright (c) 2026 QuantWall. Full text: https://opensource.org/license/mit
>
> **Disclaimer.** This content is general information and software-engineering education only. It is not financial product advice, a recommendation, or an offer to buy or sell any financial product, and it does not take into account your objectives, financial situation or needs. The author does not hold an Australian Financial Services Licence and is not a registered Commodity Trading Advisor. Trading futures and leveraged products carries a high risk of loss. Consult a licensed adviser before making any financial decision.
>
> **HYPOTHETICAL OR SIMULATED PERFORMANCE RESULTS HAVE CERTAIN LIMITATIONS. UNLIKE AN ACTUAL PERFORMANCE RECORD, SIMULATED RESULTS DO NOT REPRESENT ACTUAL TRADING. ALSO, SINCE THE TRADES HAVE NOT BEEN EXECUTED, THE RESULTS MAY HAVE UNDER-OR-OVER COMPENSATED FOR THE IMPACT, IF ANY, OF CERTAIN MARKET FACTORS, SUCH AS LACK OF LIQUIDITY. SIMULATED TRADING PROGRAMS IN GENERAL ARE ALSO SUBJECT TO THE FACT THAT THEY ARE DESIGNED WITH THE BENEFIT OF HINDSIGHT. NO REPRESENTATION IS BEING MADE THAT ANY ACCOUNT WILL OR IS LIKELY TO ACHIEVE PROFIT OR LOSSES SIMILAR TO THOSE SHOWN.**
>
> All data shown is synthetic or publicly available sample data. Features are standard textbook baselines (lagged returns, ATR, RSI, moving averages, volume). No proprietary signal is disclosed or implied.

# Why trust this channel? Every number in the video

Each cell reproduces one claim from the video from a seeded simulation of pure noise (the same functions the animations use). Change the inputs and see how the conclusions move. Requirements: `numpy`.

In [ ]:
import numpy as np

SQ = np.sqrt(252)

## 1. Channel A: post only the winners
Noise strategies (1% a day, no drift, 3 years each). A strategy is "posted" when its Sharpe ratio is above 1. How many tries does it take to fill twelve posts?

In [ ]:
def channel_a(seed=1, posts=12, days=756, cut=1.0):
    """Noise strategies (1% a day, no drift, 3 years each), 'posted' when the Sharpe is above `cut`.
    Returns every try's Sharpe, the indices of the posted ones, and the posted equity curves."""
    rng = np.random.default_rng(seed)
    sharpes, curves, posted = [], [], []
    while len(posted) < posts:
        r = rng.normal(0, 0.01, days)
        s = float(r.mean() / r.std() * SQ)
        if s > cut:
            posted.append(len(sharpes))
            curves.append(np.cumsum(r) * 100)
        sharpes.append(s)
    return np.array(sharpes), posted, curves

sharpes, posted, curves = channel_a()
print(f"tries: {len(sharpes)}   posted: {len(posted)}   never shown: {len(sharpes) - len(posted)}")
print(f"posted Sharpe, average: {sharpes[posted].mean():.2f}   all tries, average: {sharpes.mean():+.2f}")
# A Sharpe estimate over 3 years has a standard error of about 1/sqrt(3) = 0.58, so roughly 4% of noise strategies clear 1.
print(f"share of tries above 1: {np.mean(sharpes > 1):.1%}")

## 2. The best of N tries
N noise strategies with 10 years of monthly returns; keep the best. Repeated 400 times for each N.

In [ ]:
def best_of_n(seed=5, ns=(1, 10, 100, 1000), months=120, reps=400):
    """Best Sharpe among N noise strategies (10 years monthly), `reps` times for each N."""
    rng = np.random.default_rng(seed)
    out = {}
    for n in ns:
        x = rng.normal(0, 1, (reps, n, months))
        out[n] = (x.mean(axis=2) / x.std(axis=2) * np.sqrt(12)).max(axis=1)
    return out

for n, v in best_of_n().items():
    print(f"N = {n:5,d}: average best Sharpe {v.mean():+.2f}")

## 3. The research log, in proportions
The wall on screen is 100 tiles: about a fifth stopped by costs, about seven in ten failed one honest test, about one in ten passed. (Proportions only: no exact counts.)

In [ ]:
def research_log(seed=2):
    """A 10 x 10 wall in proportions: 20 tiles stopped by costs, 70 failed one honest test, 10 passed."""
    kinds = np.array([0] * 20 + [1] * 70 + [2] * 10)
    return np.random.default_rng(seed).permutation(kinds)

print(dict(zip(("costs", "failed", "passed"), np.bincount(research_log()))))

## 4. Written first: the fingerprint
A hash is a fingerprint of a file. Change one character and it changes completely, so a moved goalpost is visible.

In [ ]:
import hashlib

doc = """idea: monthly trend on 40 futures markets
who pays: hedgers paying for insurance
data: synthetic, 1990-2020, point-in-time
costs: 2 ticks per side, stress x3
pass mark: one-sided t >= 2 after costs
after the look: label is final; no variants on this data
"""
edited = doc.replace("t >= 2", "t >= 1")          # one character: the goalposts move
for name, text in (("as written", doc), ("one character changed", edited)):
    print(f"{name:22s} sha256 {hashlib.sha256(text.encode()).hexdigest()[:12]}")

## 5. One look vs peeking
4,000 noise strategies, 10 years of monthly returns, a one-sided t-test at 5% (t > 1.645). Look once at the end, or check every month (or every year) from year 1 and stop at the first pass.

In [ ]:
def peeking(seed=9, paths=4000, months=120, first=12, crit=1.645):
    """Running one-sided t-statistics of noise strategies; pass rates for one look, yearly and monthly peeks."""
    x = np.random.default_rng(seed).normal(0, 1, (paths, months))
    n = np.arange(1, months + 1)
    mean = np.cumsum(x, axis=1) / n
    var = (np.cumsum(x ** 2, axis=1) - n * mean ** 2) / np.maximum(n - 1, 1)
    t = mean / np.sqrt(np.maximum(var, 1e-12) / n)
    t[:, : first - 1] = np.nan                                            # no test before the first year
    monthly = t[:, first - 1:]
    yearly = t[:, first - 1::12]
    rates = {
        "once": float(np.mean(t[:, -1] > crit)),
        "yearly": float(np.mean((yearly > crit).any(axis=1))),
        "monthly": float(np.mean((monthly > crit).any(axis=1))),
    }
    return t, rates, crit

t, rates, crit = peeking()
for name, r in rates.items():
    print(f"{name:8s} noise passes {r:.1%} of the time")

### Try this
- Peeking: check every week instead of every month (use more, shorter periods). How high does the pass rate go?
- Channel A: raise the posting bar to a Sharpe of 1.5. How many tries does it take now?

*Simulated results on synthetic data. Education only. Not financial advice.*